# Déboguer PyTorch : forme, type, gradient, stockage et device

Identifier le contrat qui échoue, puis vérifier une seule correction.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#lire-erreurs-pytorch).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Nous savons écrire une boucle ; il faut aussi savoir l’arrêter au bon endroit. Une erreur se comprend mieux sur deux lignes isolées que dans toute une expérience.

Les blocs suivants provoquent volontairement des erreurs de forme, de type et de vue, puis testent leurs corrections. Le gradient à None est observé avant backward : ce n’est pas une exception à lui seul.

| Observation | Question à poser | Correction de cette miniature |
| --- | --- | --- |
| Linear(3,2) reçoit (2,4) | Que représentent réellement les quatre colonnes ? | Fournir les trois mesures attendues, ou choisir une architecture cohérente |
| CrossEntropyLoss reçoit une cible [1.] | Est-ce un indice de classe ou une distribution ? | Dans ce mode à indices, utiliser long/int64 |
| poids.grad vaut None | A-t-on calculé une perte puis backward ? | Dériver avant de lire le gradient |
| view refuse une transposition | La vue demandée respecte-t-elle le stockage ? | reshape si une copie éventuelle convient |
| CPU/CUDA mélangés | Données et modèle ont-ils le même device ? | Les placer sur un emplacement compatible |

Une correction doit préserver le sens des données ; obtenir un programme qui ne plante plus ne suffit pas.

Le programme reste autonome sur CPU. Le conflit CPU/CUDA est seulement testé si un GPU CUDA est disponible ; aucun tenseur meta n’est nécessaire pour le simuler.

## Démonstration 1 · Provoquer une difficulté, puis vérifier la réparation

Chaque sortie correspond à une propriété déjà rencontrée dans le cours.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch
from torch import nn


### Calculer l’état suivant

La couche attend trois mesures par extrait ; l’entrée en donne quatre. Le message affiche les tailles de la multiplication impossible.

In [4]:
try:
    nn.Linear(3, 2)(torch.ones(2, 4))
except RuntimeError as erreur:
    print("FORME :", erreur)


FORME : mat1 and mat2 shapes cannot be multiplied (2x4 and 3x2)


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [5]:
print("forme corrigee :", tuple(nn.Linear(3, 2)(torch.ones(2, 3)).shape))



forme corrigee : (2, 2)


### Calculer l’état suivant

scores contient deux classes pour un extrait. [1.] est flottant ; ce mode de CrossEntropyLoss attend pourtant un indice entier par extrait.

In [6]:
scores = torch.tensor([[1., 2.]])
try:
    nn.CrossEntropyLoss()(scores, torch.tensor([1.]))
except RuntimeError as erreur:
    print("TYPE :", erreur)
cibles = torch.tensor([1], dtype=torch.long)


TYPE : expected scalar type Long but found Float


In [7]:
observer('scores', scores)
observer('cibles', cibles)

scores forme (1, 2) type torch.float32 requires_grad False
  valeurs : [[1.0, 2.0]]
cibles forme (1,) type torch.int64 requires_grad False
  valeurs : [1]


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [8]:
print("type corrige :", cibles.dtype)



type corrige : torch.int64


### Calculer l’état suivant

requires_grad=True autorise le suivi. Il ne réalise aucune dérivation tout seul : grad vaut encore None avant backward.

In [9]:
poids = torch.tensor(2., requires_grad=True)


In [10]:
observer('poids', poids)

poids forme () type torch.float32 requires_grad True
  valeurs : 2.0


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [11]:
print("gradient avant backward :", poids.grad)


gradient avant backward : None


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [12]:
perte = poids ** 2


In [13]:
observer('perte', perte)

perte forme () type torch.float32 requires_grad True
  valeurs : 4.0


### Calculer les gradients

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [14]:
perte.backward()


In [15]:
observer_entrainement({'scores': scores, 'cibles': cibles, 'poids': poids, 'perte': perte}, montrer_poids=False)

poids.grad forme () type torch.float32 requires_grad False
  valeurs : 4.0


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [16]:
print("gradient apres backward :", poids.grad.item())



gradient apres backward : 4.0


### Calculer l’état suivant

La transposition change l’ordre logique des axes. view(6) ne peut pas représenter ce parcours dans ce stockage ; reshape peut fournir une copie.

In [17]:
tableau = torch.arange(6).reshape(2, 3)
transpose = tableau.t()
try:
    transpose.view(6)
except RuntimeError as erreur:
    print("VUE :", erreur)


VUE : view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.


In [18]:
observer('tableau', tableau)
observer('transpose', transpose)

tableau forme (2, 3) type torch.int64 requires_grad False
  valeurs : [[0, 1, 2], [3, 4, 5]]
transpose forme (3, 2) type torch.int64 requires_grad False
  valeurs : [[0, 3], [1, 4], [2, 5]]


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [19]:
print("reshape possible :", transpose.reshape(6).tolist())



reshape possible : [0, 3, 1, 4, 2, 5]


### Calculer l’état suivant

Une entrée CPU ne convient pas à cette couche CUDA. La correction déplace l’entrée au même endroit ; sur CPU seul, la branche est simplement annoncée non exécutée.

In [20]:
if torch.cuda.is_available():
    couche_gpu = nn.Linear(2, 2).to("cuda")
    entree_cpu = torch.ones(1, 2)
    try:
        couche_gpu(entree_cpu)
    except RuntimeError as erreur:
        print("DEVICE :", erreur)
    print("device corrige :", tuple(couche_gpu(entree_cpu.to("cuda")).shape))
else:
    print("CPU seul : conflit CPU/CUDA non provoque.")

CPU seul : conflit CPU/CUDA non provoque.


Le message local aide à choisir une vérification précise, plutôt qu’une transformation arbitraire du tableau.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Après une transposition, choisissez la méthode qui peut aplatir ce tableau sans exiger sa contiguïté.

In [21]:
exo_x = torch.arange(6).reshape(2,3).transpose(0,1)
exo_methode = ...  # Remplacer les points par votre réponse.
if exo_methode is Ellipsis or exo_methode is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    exo_resultat=getattr(exo_x,exo_methode)(-1)
    assert exo_resultat.tolist() == [0,3,1,4,2,5]
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

reshape peut copier si nécessaire ; view impose une compatibilité de strides.

</details>

### Exercice 2

Corrigez le type d’indices de classe stockés par erreur en float32.

In [22]:
exo_cibles = torch.tensor([0.,1.],dtype=torch.float32)
exo_correction = ...  # Remplacer les points par votre réponse.
if exo_correction is Ellipsis or exo_correction is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_correction.dtype == torch.int64
    assert exo_correction.tolist() == [0,1]
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

La conversion porte sur des indices déjà entiers dans leur sens. Une cible multi-étiquettes demanderait un autre contrat.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_x = torch.arange(6).reshape(2,3).transpose(0,1)
exo_methode = "reshape"
if exo_methode is Ellipsis or exo_methode is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    exo_resultat=getattr(exo_x,exo_methode)(-1)
    assert exo_resultat.tolist() == [0,3,1,4,2,5]
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_cibles = torch.tensor([0.,1.],dtype=torch.float32)
exo_correction = exo_cibles.to(torch.long)
if exo_correction is Ellipsis or exo_correction is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_correction.dtype == torch.int64
    assert exo_correction.tolist() == [0,1]
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [23]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()